In [ ]:
# imports
import os
import openai
import glob
import yt_dlp as yt
import docarray
from yt_dlp import DownloadError
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# An example YouTube tutorial video
youtube_url = "https://www.youtube.com/watch?v=aqzxYofJ_ck"
# Directory to store the downloaded video
output_dir = "files/audio/"

openai.api_key = os.getenv("OPENAI_API_KEY")

# Config for youtube-dl
ydl_config = {
    "format": "bestaudio/best",
    "postprocessors": [
        {
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }
    ],
    "outtmpl": os.path.join(output_dir, "%(title)s.%(ext)s"),
    "verbose": True
}

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"Downloading audio from {youtube_url}...")

try:
    with yt.YoutubeDL(ydl_config) as ydl:
        ydl.download([youtube_url])
except DownloadError as e:
    print(f"Error downloading video: {e}")

In [ ]:
audio_files = glob.glob(os.path.join(output_dir, "*.mp3"))
print(f"Downloaded audio files: {audio_files}")

In [ ]:
output_dir = "files/transcripts/"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

model = "whisper-1"
print("converting audio to text using OpenAI Whisper API...")
for audio_file in audio_files:
    print(f"Transcribing {audio_file}...")
    with open(audio_file, "rb") as audio:
        response = openai.audio.transcriptions.create(model=model, file=audio)

transcript = response["text"]
transcript_file = os.path.join(output_dir, os.path.basename(audio_file).replace(".mp3", ".txt"))
with open(transcript_file, "w") as f:
    f.write(transcript)
print(f"Transcript saved to {transcript_file}")

In [ ]:
from langchain.document_loaders import TextLoader   
loader = TextLoader(transcript_file)
documents = loader.load()
print(f"Loaded {len(documents)} documents from transcript.")

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")
tokens = tokenizer.encode(transcript)
print(f"Transcript token count: {len(tokens)}")

In [ ]:
# from langchain.chains import RetrievalQA
from langchain_classic.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
# from langchain.vectorstores import DocArrayInMemorySearch
from langchain_core.vectorstores import InMemoryVectorStore
# from langchain_community.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings

In [ ]:
db = InMemoryVectorStore.from_documents(documents, OpenAIEmbeddings())
retriever = db.as_retriever()
qa_chain = RetrievalQA.from_chain_type(llm=ChatOpenAI(), retriever=retriever)
query = "What are the key steps to create a YouTube agent?"
answer = qa_chain.run(query)
print(f"Answer: {answer}")

In [ ]:
llm = ChatOpenAI(temperature=0.7)
stuff = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True, verbose=True)

query = "What is the main message about?"
result = stuff(query) # stuff.run(query)
print(f"Answer: {result['result']}")
print("Source documents:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

In [ ]:
# Modern Alternative (Recommended)  
# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_openai import ChatOpenAI

# 1. Setup your LLM and Retriever
# model = ChatOpenAI()
# retriever = ... # Your configured retriever

# 2. Define prompt
# system_prompt = "Use the following context to answer the question: {context}"
# prompt = ChatPromptTemplate.from_messages([
#     ("system", system_prompt),
#     ("human", "{input}"),
# ])

# 3. Combine document chains and retriever
# question_answer_chain = create_stuff_documents_chain(model, prompt)
# rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 4. Invoke
# response = rag_chain.invoke({"input": "Your question here"})
